# 비유의성 진단: 표본과 결과변수

**질문:** 매출이 관측되는 표본의 구조와 결과변수 정의가 공연 활동의 관계를 포착하는 데 어떤 제약을 만드는가?

현재 확정 입력만 읽는 오프라인 진단이다. 유의한 사양을 찾거나 기존 회귀를 변경하지 않는다. 결측 메커니즘을 식별하는 자료가 없으므로 아래의 편중을 “비유의성의 입증된 원인”으로 해석하지 않는다. 노출 측정 및 추정 정밀도 진단과 함께 읽는다.

## 1. 입력과 분석 단위

종로구·마포구, 외식업 10종, 2023~2025년 상권×업종×분기의 완전 격자 18,120행이다. 이는 실제 점포가 존재하는 집단 18,120개를 뜻하지 않는다. 일반점포와 프랜차이즈를 합친 전체점포 수를 사용해 미관측과 0점포를 구분한다.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import hashlib, json
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'final').is_dir())
OUT = ROOT/'reports/final/null_diagnostics'
OUT.mkdir(parents=True, exist_ok=True)
inputs = {'panel': ROOT/'data/analysis/final/food_industry_panel.csv',
          'membership': ROOT/'reports/final/09_model_sample_membership.csv',
          'window_flow': ROOT/'reports/final/time_windows/11_sample_flow.csv'}
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
hashes = {p.relative_to(ROOT).as_posix():sha(p) for p in inputs.values()}
reference = json.loads((ROOT/'reports/final/09_analysis_summary.json').read_text(encoding='utf-8'))
assert sha(inputs['panel']) == reference['input_sha256']
panel = pd.read_csv(inputs['panel'], dtype={'area_id':str,'industry_id':str,'quarter':str})
keys=['area_id','industry_id','quarter']
assert not panel.duplicated(keys).any()
assert len(panel)==18120
panel['sales_observed']=panel.total_sales.notna()
print('범위:',panel.area_id.nunique(),'상권 ×',panel.industry_id.nunique(),'업종 ×',panel.quarter.nunique(),'분기')

범위: 151 상권 × 10 업종 × 12 분기


## 2. 매출이 관측되는 집단의 크기가 다르다

점포 수가 작은 집단과 업종별로 매출 제공률을 확인한다. 실제 영업하지 않는 집단까지 분모에 포함하는 전체 격자 제공률과, 점포가 있는 집단의 제공률을 구분한다.

In [2]:
def coverage(frame, group):
    result=frame.groupby(group, observed=True, dropna=False).agg(rows=('sales_observed','size'),
                 observed=('sales_observed','sum')).reset_index()
    result['coverage_pct']=100*result.observed/result.rows
    return result
store = coverage(panel, 'total_store_size_group')
industry=coverage(panel,['industry_id','industry'])
positive=coverage(panel.loc[panel.total_store_count.gt(0)],['industry_id','industry'])
positive=positive.rename(columns={'rows':'positive_store_rows','observed':'positive_store_observed','coverage_pct':'positive_store_coverage_pct'})
industry=industry.merge(positive,on=['industry_id','industry'],validate='one_to_one')
store.to_csv(OUT/'sample_store_coverage.csv',index=False,encoding='utf-8-sig')
industry.to_csv(OUT/'sample_industry_coverage.csv',index=False,encoding='utf-8-sig')
display(store)
display(industry)


,total_store_size_group,rows,observed,coverage_pct
0,0,87,0,0.000000
1,11~50,3109,3040,97.780637
2,1~2,4430,0,0.000000
3,3~5,2718,1811,66.629875
4,51+,854,854,100.000000
5,6~10,2114,1956,92.526017
6,미제공,4808,0,0.000000


,industry_id,industry,rows,observed,coverage_pct,positive_store_rows,positive_store_observed,positive_store_coverage_pct
0,CS100001,한식음식점,1812,1489,82.174393,1681,1489,88.578227
1,CS100002,중식음식점,1812,464,25.607064,1064,464,43.609023
2,CS100003,일식음식점,1812,479,26.434879,1132,479,42.314488
3,CS100004,양식음식점,1812,660,36.423841,1263,660,52.256532
4,CS100005,제과점,1812,534,29.470199,1308,534,40.825688
5,CS100006,패스트푸드점,1812,446,24.613687,1183,446,37.700761
6,CS100007,치킨전문점,1812,480,26.490066,1114,480,43.087971
7,CS100008,분식전문점,1812,857,47.295806,1388,857,61.743516
8,CS100009,호프-간이주점,1812,944,52.097130,1410,944,66.950355
9,CS100010,커피-음료,1812,1308,72.185430,1682,1308,77.764566


**해석:** 미제공 매출을 0으로 채우지 않는다. 전체점포 수가 양수인 집단에서도 제공률 차이가 남는다면 표본이 상권·업종의 전 범위를 고르게 대표하지 않는다는 관측 증거다. 그러나 제외된 매출을 모르므로 누락으로 회귀계수가 어느 방향으로 얼마나 달라졌는지는 계산할 수 없다.

## 3. 업종이 늘어도 같은 상권을 더 많이 관측한 것은 아니다

주모형과 시간창 모형의 실제 표본 수를 분리하고, 업종별 표본이 어느 정도 겹치는지 계산한다. 6개 시간대는 같은 상권·업종·분기에 반복된 행이므로 서로 독립적인 6개 실험으로 세지 않는다.

In [3]:
members = pd.read_csv(inputs['membership'],dtype={k:str for k in keys})
main = members.query("scenario == 'main_500' and outcome == 'log_sales'")[keys].drop_duplicates()
assert len(main)==7646
observed = panel[panel.sales_observed]
all10_grid = observed.groupby(['area_id','quarter']).industry_id.nunique().eq(10)
main_all10 = main.groupby(['area_id','quarter']).industry_id.nunique().eq(10)
common_keys=main_all10[main_all10].reset_index()[['area_id','quarter']]
common_rows=main.merge(common_keys,on=['area_id','quarter'],validate='many_to_one')
window_flow=pd.read_csv(inputs['window_flow'])
window90=window_flow.loc[window_flow.coverage_threshold.eq(90)]
summary={
    'grid_rows':len(panel),'sales_rows':int(panel.sales_observed.sum()),
    'grid_sales_coverage_pct':100*panel.sales_observed.mean(),
    'positive_store_rows':int(panel.total_store_count.gt(0).sum()),
    'positive_store_sales_coverage_pct':100*panel.loc[panel.total_store_count.gt(0),'sales_observed'].mean(),
    'main_model_rows':len(main),'main_model_areas':main.area_id.nunique(),
    'all10_observed_area_quarters':int(all10_grid.sum()),
    'all10_main_area_quarters':len(common_keys),'all10_main_areas':common_keys.area_id.nunique(),
    'all10_main_rows':len(common_rows),'all10_main_share_pct':100*len(common_rows)/len(main),
    'window90_timebin_rows':int(window90.timebin_rows.sum()),
    'window90_base_area_industry_quarters':int(window90.area_industry_quarters.sum()),
    'window_base_retention_vs_main_pct':100*window90.area_industry_quarters.sum()/len(main),
}
main_counts=main.groupby('industry_id').agg(model_rows=('quarter','size'),model_areas=('area_id','nunique')).reset_index()
industry=industry.merge(main_counts,on='industry_id',validate='one_to_one')
industry.to_csv(OUT/'sample_industry_coverage.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame([summary]).T.rename(columns={0:'수치'}))

,수치
grid_rows,18120.000000
sales_rows,7661.000000
grid_sales_coverage_pct,42.279249
positive_store_rows,13225.000000
positive_store_sales_coverage_pct,57.928166
main_model_rows,7646.000000
main_model_areas,134.000000
all10_observed_area_quarters,171.000000
all10_main_area_quarters,170.000000
all10_main_areas,16.000000


**해석:** 업종별 추정치가 다르면 음식 종류와 상권 구성의 차이가 함께 반영될 수 있다. 모든 업종이 관측되는 공통표본으로 바꾸면 구성은 맞출 수 있지만 많은 관측을 잃는다. 공통표본만 쓰는 것이 자동으로 더 정확하다는 뜻은 아니다.

## 4. 공연 활동 수준과 매출 제공의 관계

노출 구간은 분석 전에 정한 0개·1~10개·11~50개·51개 이상이다. 이 표는 제공 여부의 기술통계이며 누락의 인과관계나 공연 효과 회귀가 아니다.

In [4]:
panel['exposure_group']=pd.cut(panel.known_performance_count_500m,
    bins=[-0.1,0,10,50,np.inf],labels=['0개','1~10개','11~50개','51개 이상'])
exposure=coverage(panel,'exposure_group')
exposure_positive=coverage(panel.loc[panel.total_store_count.gt(0)],'exposure_group').rename(
    columns={'rows':'positive_store_rows','observed':'positive_store_observed','coverage_pct':'positive_store_coverage_pct'})
exposure=exposure.merge(exposure_positive,on='exposure_group',validate='one_to_one')
exposure.to_csv(OUT/'sample_exposure_coverage.csv',index=False,encoding='utf-8-sig')
district=coverage(panel,['district','industry_id','industry'])
district.to_csv(OUT/'sample_district_coverage.csv',index=False,encoding='utf-8-sig')
display(exposure)

,exposure_group,rows,observed,coverage_pct,positive_store_rows,positive_store_observed,positive_store_coverage_pct
0,0개,3250,1117,34.369231,2296,1117,48.649826
1,1~10개,4830,1541,31.904762,3103,1541,49.661618
2,11~50개,5080,1979,38.956693,3628,1979,54.547960
3,51개 이상,4960,3024,60.967742,4198,3024,72.034302


## 5. 야간 매출 비중이 낮아져도 야간 매출액은 늘 수 있다

야간 비중은 `야간 매출액 / 총매출액`이다. 같은 상권·업종의 인접 분기에서 실제 야간 매출액 변화와 비중 변화의 방향이 일치하는지 확인한다. 계절·물가·공연 등 원인을 분리한 효과가 아니라 지표가 서로 다른 현상을 측정한다는 기술통계다.

In [5]:
p=panel.sort_values(['area_id','industry_id','quarter']).copy()
p['q_index']=p.year*4+p.quarter_number
for col in ['q_index','total_sales','night_sales','night_share']:
    p['prev_'+col]=p.groupby(['area_id','industry_id'])[col].shift()
adj=p.loc[p.q_index.sub(p.prev_q_index).eq(1)&p.total_sales.notna()&p.prev_total_sales.notna()
          &p.night_sales.notna()&p.prev_night_sales.notna()&p.night_share.notna()&p.prev_night_share.notna()].copy()
adj['night_amount_change']=adj.night_sales-adj.prev_night_sales
adj['night_share_change_pp']=100*(adj.night_share-adj.prev_night_share)
for col in ['night_amount_change','night_share_change_pp']:
    adj[col+'_direction']=np.select([adj[col]>0,adj[col]<0],['증가','감소'],default='불변')
matrix=adj.groupby(['night_amount_change_direction','night_share_change_pp_direction']).size().reset_index(name='adjacent_pairs')
matrix['share_pct']=100*matrix.adjacent_pairs/len(adj)
matrix.to_csv(OUT/'sample_night_amount_vs_share.csv',index=False,encoding='utf-8-sig')
summary['adjacent_observed_pairs']=len(adj)
summary['night_amount_increase_share_decrease_pairs']=int((adj.night_amount_change.gt(0)&adj.night_share_change_pp.lt(0)).sum())
summary['night_amount_increase_share_decrease_pct']=100*summary['night_amount_increase_share_decrease_pairs']/len(adj)
summary['opposite_night_amount_share_direction_pct']=100*(adj.night_amount_change*adj.night_share_change_pp<0).mean()
display(matrix)

,night_amount_change_direction,night_share_change_pp_direction,adjacent_pairs,share_pct
0,감소,감소,2708,38.941616
1,감소,불변,1,0.014380
2,감소,증가,694,9.979868
3,불변,불변,613,8.815070
4,불변,증가,1,0.014380
5,증가,감소,646,9.289617
6,증가,증가,2291,32.945068


**해석:** 야간 매출 비중의 비유의성은 야간 매출액 효과가 없다는 검정이 아니다. 공연이 모든 시간대 매출을 비슷하게 늘리면 비중은 거의 변하지 않을 수 있다. 현재 시간창 모형도 상권×업종×분기 고정효과를 사용하므로 분기 전체에 공통인 매출 수준 변화는 제거한다. 따라서 총매출 증가와 시간대 재배분은 별도 질문으로 명시해야 한다.

## 6. 증거 저장과 한계

이 진단은 자료 제공·표본 중첩·결과변수 정의를 설명한다. 미제공 매출, 실제 관객 수, 공연별 실현 소비가 없으므로 각각이 비유의성에 기여한 비율은 추정할 수 없다. 모델을 바꾸어 p값을 줄인 작업이 아니다.

In [6]:
summary['source_sha256']=hashes
summary['network_requests']=0
summary['existing_models_modified']=False
summary['created_at']=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
assert all(sha(ROOT/p)==h for p,h in hashes.items())
(OUT/'sample_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([{'검사':'키 중복 없음·18,120행','통과':True},
                      {'검사':'기존 주모형 7,646행 일치','통과':len(main)==7646},
                      {'검사':'입력 SHA256 불변','통과':True}]))

,검사,통과
0,"키 중복 없음·18,120행",True
1,"기존 주모형 7,646행 일치",True
2,입력 SHA256 불변,True
